# Construcción del pipeline

Este notebook muestra, paso a paso, cómo se arma el pipeline del modelo: cada transformación se prueba por separado y al final se encadenan todas con el modelo.

**¿Por qué un pipeline?**

- Recibe los datos crudos, tal como vienen del CSV, y devuelve la predicción. No hay pasos manuales intermedios que se puedan olvidar.
- El entrenamiento y la predicción ejecutan exactamente el mismo código.
- Todo lo que se aprende de los datos (por ejemplo, las categorías del *one-hot encoding*) se aprende solo del conjunto de entrenamiento, lo que evita fugas de información hacia el conjunto de prueba.

Las decisiones de limpieza y selección de variables vienen de `eda/eda.ipynb`. El código de los transformadores está en `transformadores.py`.

In [1]:
import inspect

import pandas as pd
from sklearn.model_selection import train_test_split

from transformadores import LimpiarTotalCharges, EliminarColumnas, CrearCantidadServicios, MapearBinarias

## 1. Datos y partición

In [2]:
df = pd.read_csv("../data/telco_churn.csv")
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(X_train.columns.tolist())

(5634, 20) (1409, 20)
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


## 2. Transformadores propios

Cada transformador es una clase que hereda de `BaseEstimator` y `TransformerMixin`, de modo que scikit-learn la puede usar dentro de un `Pipeline`.

### Paso 1: limpiar `TotalCharges`

La columna llega como texto porque los clientes recién llegados (11 en todo el dataset) tienen un espacio en blanco en lugar de un número. Se convierte a número y los vacíos pasan a 0.

In [3]:
print(inspect.getsource(LimpiarTotalCharges))

class LimpiarTotalCharges(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalCharges'] = pd.to_numeric(X['TotalCharges'], errors='coerce')
        X['TotalCharges'] = X['TotalCharges'].fillna(0)
        return X



In [4]:
limpiador = LimpiarTotalCharges()
X_paso1 = limpiador.fit_transform(X_train)

print(X_paso1["TotalCharges"].dtype)            # debe ser float64
print(X_paso1["TotalCharges"].isnull().sum())   # debe ser 0

float64
0


### Paso 2: eliminar columnas

Se descartan el identificador del cliente y las tres variables que no mostraron relación con la cancelación en el EDA.

In [5]:
print(inspect.getsource(EliminarColumnas))

class EliminarColumnas(BaseEstimator, TransformerMixin):
    def __init__(self, columnas_a_eliminar):
        self.columnas_a_eliminar = columnas_a_eliminar
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.columnas_a_eliminar, errors='ignore')



In [6]:
eliminador = EliminarColumnas(["gender", "PhoneService", "MultipleLines", "customerID"])
X_paso2 = eliminador.fit_transform(X_paso1)

print(X_paso2.columns.tolist())
print(X_paso2.shape)

['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']
(5634, 16)


### Paso 3: crear `cantidad_servicios`

Cuatro servicios adicionales, muy relacionados entre sí, se resumen en una sola variable: cuántos tiene contratados el cliente.

In [7]:
print(inspect.getsource(CrearCantidadServicios))

class CrearCantidadServicios(BaseEstimator, TransformerMixin):
    def __init__(self, columnas_servicios):
        self.columnas_servicios = columnas_servicios
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['cantidad_servicios'] = (X[self.columnas_servicios] == 'Yes').sum(axis=1)
        return X.drop(columns=self.columnas_servicios)



In [8]:
columnas_servicios = ["OnlineBackup", "DeviceProtection", "StreamingTV", "StreamingMovies"]

creador_servicios = CrearCantidadServicios(columnas_servicios)
X_paso3 = creador_servicios.fit_transform(X_paso2)

print(X_paso3.columns.tolist())
print(X_paso3["cantidad_servicios"].value_counts().sort_index())

['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'InternetService', 'OnlineSecurity', 'TechSupport', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'cantidad_servicios']
cantidad_servicios
0    2037
1    1032
2    1018
3     933
4     614
Name: count, dtype: int64


### Paso 4: convertir a 0 y 1 las variables de sí/no

`No internet service` se trata como `No`: el cliente no tiene ese servicio.

In [9]:
print(inspect.getsource(MapearBinarias))

class MapearBinarias(BaseEstimator, TransformerMixin):
    def __init__(self, columnas_binarias):
        self.columnas_binarias = columnas_binarias
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        mapeo = {'No': 0, 'Yes': 1, 'No internet service': 0}
        for col in self.columnas_binarias:
            X[col] = X[col].map(mapeo)
        return X



In [10]:
binarias = ["Partner", "Dependents", "PaperlessBilling", "OnlineSecurity", "TechSupport"]

mapeador = MapearBinarias(binarias)
X_paso4 = mapeador.fit_transform(X_paso3)

print(X_paso4[binarias].head())
print(X_paso4[binarias].isnull().sum())   # no debe quedar ningún valor sin mapear

      Partner  Dependents  PaperlessBilling  OnlineSecurity  TechSupport
3738        0           0                 0               0            0
3151        1           1                 0               1            0
4860        1           1                 0               1            1
3867        1           0                 1               0            0
3810        1           1                 0               0            0
Partner             0
Dependents          0
PaperlessBilling    0
OnlineSecurity      0
TechSupport         0
dtype: int64


## 3. Codificación de variables nominales

Las variables con más de dos categorías se codifican con *one-hot encoding*. `drop="first"` evita columnas redundantes y `handle_unknown="ignore"` evita errores si en producción aparece una categoría nueva. El resto de columnas pasa sin cambios.

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

nominales = ["InternetService", "Contract", "PaymentMethod"]

preprocessor = ColumnTransformer(
    transformers=[
        ("nominales", OneHotEncoder(drop="first", handle_unknown="ignore"), nominales)
    ],
    remainder="passthrough",
)

X_paso5 = preprocessor.fit_transform(X_paso4)

print(X_paso5.shape)
print([nombre.split("__", 1)[1] for nombre in preprocessor.get_feature_names_out()])

(5634, 17)
['InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'OnlineSecurity', 'TechSupport', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'cantidad_servicios']


## 4. Pipeline completo

Se encadenan los cinco pasos de preprocesamiento con el modelo. El modelo final es un **Random Forest**, elegido en `comparaciones/comparacion_modelos.ipynb` por tener el mejor F1 y ROC-AUC; sus hiperparámetros salen de la búsqueda hecha allí.

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

pipeline_final = Pipeline([
    ("limpiar_totalcharges", LimpiarTotalCharges()),
    ("eliminar_columnas", EliminarColumnas(["gender", "PhoneService", "MultipleLines", "customerID"])),
    ("crear_cantidad_servicios", CrearCantidadServicios(columnas_servicios)),
    ("mapear_binarias", MapearBinarias(binarias)),
    ("encoding_nominales", preprocessor),
    ("modelo", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

pipeline_final.fit(X_train, y_train)
print([nombre for nombre, _ in pipeline_final.steps])

['limpiar_totalcharges', 'eliminar_columnas', 'crear_cantidad_servicios', 'mapear_binarias', 'encoding_nominales', 'modelo']


El pipeline recibe los datos crudos del conjunto de prueba (20 columnas, con `TotalCharges` como texto) y devuelve la predicción:

In [13]:
from sklearn.metrics import classification_report, roc_auc_score

pred = pipeline_final.predict(X_test)
print(classification_report(y_test, pred, target_names=["No cancela", "Cancela"]))
print(f"ROC-AUC: {roc_auc_score(y_test, pipeline_final.predict_proba(X_test)[:, 1]):.3f}")

              precision    recall  f1-score   support

  No cancela       0.90      0.76      0.83      1035
     Cancela       0.54      0.76      0.63       374

    accuracy                           0.76      1409
   macro avg       0.72      0.76      0.73      1409
weighted avg       0.80      0.76      0.77      1409



ROC-AUC: 0.842


## 5. Siguiente paso

`modelo.py` contiene este mismo pipeline en forma de script: lo entrena y lo guarda en `pipeline_churn.pkl`.

```bash
python modelo.py
```

El modelo guardado se evalúa y se explica en `pruebas/evaluacion_shap.ipynb`.